# Notebook 2: Semantic Filtering & CPC Validation Audit
Loads the deduplicated DOCDB patent family baseline (`../flying_drones_baseline_prod_final.parquet`) and pre-computed `graziasveva93/sbert-adapt-ub` embeddings (`../flying_drones_embeddings_final.npy`), computes cosine similarity using both **Strategy A (Seed Centroid)** and **Strategy B (Technical Description Reference Vector)**, runs a **Threshold Sensitivity & Disagreement Sweep** alongside a **CPC Subclass Migration Audit**, and exports the finalized Stage 2 dataset.

In [10]:
!pip install -q sentence-transformers pandas pyarrow numpy torch scikit-learn


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [11]:
import os
import re
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Load Baseline Corpus & Verify Embedding Alignment

In [12]:
# 1. Load deduplicated DOCDB family Parquet baseline
parquet_path = "../flying_drones_baseline_prod_final.parquet"
df = pd.read_parquet(parquet_path)
print(f"Loaded Parquet baseline: {len(df):,} patent families")

# 2. Apply exact text sanitization matching the embedding generation script
df['appln_title'] = df['appln_title'].fillna('').str.strip()
df['appln_abstract'] = df['appln_abstract'].fillna('').str.strip()
df['full_text'] = df['appln_title'] + ". " + df['appln_abstract']

short_mask = df['full_text'].str.strip().str.len() <= 10
df = df[~short_mask].copy().reset_index(drop=True)
print(f"Sanitized corpus ready: {len(df):,} patent families")

# 3. Load pre-computed embeddings and verify strict 1:1 row alignment
corpus_embeddings = np.load("../flying_drones_embeddings_final.npy")
assert len(df) == corpus_embeddings.shape[0], (
    f"Row mismatch: {len(df)} DataFrame rows vs {corpus_embeddings.shape[0]} embeddings"
)
print(f"Embeddings verified: {corpus_embeddings.shape}")

Loaded Parquet baseline: 95,426 patent families
Sanitized corpus ready: 95,426 patent families
Embeddings verified: (95426, 384)


In [13]:
# 4. Verify right-truncation date window (2000-01-01 to 2022-12-31)
# Already applied in Notebook 1; this guard keeps df and corpus_embeddings aligned if raw data is ever loaded
MIN_DATE = pd.Timestamp("2000-01-01")
MAX_DATE = pd.Timestamp("2023-01-01")

date_col = 'earliest_filing_date' if 'earliest_filing_date' in df.columns else 'appln_filing_date'
filing_dates = pd.to_datetime(df[date_col], errors='coerce')
date_mask = (filing_dates >= MIN_DATE) & (filing_dates < MAX_DATE)

if not date_mask.all():
    df = df[date_mask].copy().reset_index(drop=True)
    corpus_embeddings = corpus_embeddings[date_mask.values]
    print(f"Applied date truncation (< {MAX_DATE.date()}): {len(df):,} families | Embeddings: {corpus_embeddings.shape}")
else:
    print(f"Date window verified (< {MAX_DATE.date()}): all {len(df):,} families within [2000, 2022]")

Date window verified (< 2023-01-01): all 95,426 families within [2000, 2022]


## 2. Compute Semantic Similarity Scores (Strategy A vs. Strategy B)
Computes two complementary cosine similarity scores using `graziasveva93/sbert-adapt-ub`:
- **Strategy A (`sim_score_centroid`):** Mean normalized embedding of 15 gold-standard `B64U` UAV patents (captures authentic patent phrasing).
- **Strategy B (`sim_reference_vector`):** Embedding of a comprehensive domain definition (`TECH_DESCRIPTION`) covering fixed-wing, VTOL, rotorcraft, propulsion, control, docking, payloads, and UTM (eliminates seed-selection bias).

In [14]:
# Load the exact same domain-adapted SBERT model used to encode corpus_embeddings
SBERT_MODEL = "graziasveva93/sbert-adapt-ub"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(SBERT_MODEL, device=device)
print(f"Loaded '{SBERT_MODEL}' on {device}")

# --- Strategy A: Seed-Centroid Cosine Similarity ---
seed_candidates = df[
    df['cpc_codes'].str.contains('B64U', na=False) &
    df['appln_title'].str.contains('unmanned aerial vehicle|quadcopter|multirotor', case=False, na=False)
].head(15)

seed_embeddings = model.encode(
    seed_candidates['full_text'].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True
)
centroid_vector = np.mean(seed_embeddings, axis=0, keepdims=True)
centroid_vector = centroid_vector / np.linalg.norm(centroid_vector)

df['sim_score_centroid'] = cosine_similarity(corpus_embeddings, centroid_vector).flatten()

print("\nStrategy A (Seed Centroid) Quantiles:")
print(df['sim_score_centroid'].quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded 'graziasveva93/sbert-adapt-ub' on cpu

Strategy A (Seed Centroid) Quantiles:
0.10    0.1259
0.25    0.4881
0.50    0.6550
0.75    0.7859
0.90    0.8768
0.95    0.9115
0.99    0.9494
Name: sim_score_centroid, dtype: float64


In [15]:
# --- Strategy B: Domain Technical Description Reference Vector ---
TECH_DESCRIPTION = """
Unmanned aerial vehicles (UAV), drones, and remotely piloted aircraft systems: aerial vehicles that
do not carry a human operator and are guided autonomously or remotely operated using automated
functions, including multirotors, quadcopters, rotorcraft, VTOL, and fixed-wing unmanned aircraft,
expendable or recoverable, carrying civil or military payloads. Includes equipment specially adapted
for unmanned aeronautical use: airframe and constructional aspects, wings and rotors, propulsion and
power supply, flight and attitude control systems, launching, take-off, landing or docking
arrangements, payload mounting, and unmanned aircraft traffic control.
""".strip()

reference_vector = model.encode(
    TECH_DESCRIPTION,
    convert_to_numpy=True,
    normalize_embeddings=True
).reshape(1, -1)

df['sim_reference_vector'] = cosine_similarity(corpus_embeddings, reference_vector).flatten()
df['sim_mean'] = 0.5 * (df['sim_score_centroid'] + df['sim_reference_vector'])

print("Strategy B (Reference Description Vector) Quantiles:")
print(df['sim_reference_vector'].quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).round(4))

Strategy B (Reference Description Vector) Quantiles:
0.10    0.1259
0.25    0.4976
0.50    0.6650
0.75    0.7802
0.90    0.8590
0.95    0.8905
0.99    0.9292
Name: sim_reference_vector, dtype: float64


## 3. Threshold Calibration: Side-by-Side Comparison & Sensitivity Sweep
Compares score separation between core UAV classification families (`B64U`, `B64C 39/02`) and keyword-only families, evaluates yield and CPC purity across thresholds (`0.68` to `0.84`), and inspects boundary disagreements between Strategy A and Strategy B.

In [16]:
# 1. Core-code vs. keyword-only score separation
core_strict = r'(?:B64U|B64C\s*39/02)'
df['in_core_code'] = (
    df['cpc_codes'].fillna('').str.contains(core_strict, regex=True) |
    df['ipc_codes'].fillna('').str.contains(core_strict, regex=True)
)

print("=== 1. SCORE SEPARATION: CORE UAV CODES vs. KEYWORD-ONLY FAMILIES ===")
for col in ['sim_score_centroid', 'sim_reference_vector']:
    stats = df.groupby('in_core_code')[col].quantile([0.10, 0.25, 0.50, 0.75, 0.90]).unstack().round(3)
    stats.insert(0, 'n', df.groupby('in_core_code')[col].count())
    print(f"\n[{col}]")
    print(stats.to_string())

print(f"\nPearson correlation between Strategy A and Strategy B: {df['sim_score_centroid'].corr(df['sim_reference_vector']):.3f}")

# 2. Side-by-side threshold sweep (Centroid vs. Reference Vector vs. Mean Ensemble)
aero_pattern = r'\b(B64U|B64C|B64D|G05D|B64F)\b'
noise_pattern = r'\b(G06F|G06V|H04W|G06Q)\b'

has_aero = df['cpc_codes'].fillna('').str.contains(aero_pattern, regex=True)
has_noise = df['cpc_codes'].fillna('').str.contains(noise_pattern, regex=True)

rows = []
for th in np.arange(0.68, 0.85, 0.02):
    th = round(th, 2)
    m_cent = df['sim_score_centroid'] >= th
    m_ref  = df['sim_reference_vector'] >= th
    m_mean = df['sim_mean'] >= th
    jaccard = (m_cent & m_ref).sum() / max((m_cent | m_ref).sum(), 1)
    rows.append({
        'Threshold': th,
        'Cent_N': int(m_cent.sum()),
        'Cent_Aero%': round(has_aero[m_cent].mean() * 100, 1),
        'Cent_Noise%': round(has_noise[m_cent].mean() * 100, 1),
        'Ref_N': int(m_ref.sum()),
        'Ref_Aero%': round(has_aero[m_ref].mean() * 100, 1),
        'Ref_Noise%': round(has_noise[m_ref].mean() * 100, 1),
        'Mean_N': int(m_mean.sum()),
        'Mean_Aero%': round(has_aero[m_mean].mean() * 100, 1),
        'Jaccard': round(jaccard, 2),
    })

print("\n=== 2. THRESHOLD SWEEP COMPARISON ===")
print(pd.DataFrame(rows).to_string(index=False))

# 3. Inspect boundary disagreements around 0.78 (where one method accepts and the other rejects)
only_ref = df[(df['sim_reference_vector'] >= 0.78) & (df['sim_score_centroid'] < 0.74)]
only_cent = df[(df['sim_score_centroid'] >= 0.78) & (df['sim_reference_vector'] < 0.74)]

print(f"\n=== 3. DISAGREEMENTS (Ref >= 0.78 but Centroid < 0.74 | Total: {len(only_ref):,}) ===")
for _, r in only_ref.sample(min(5, len(only_ref)), random_state=42).iterrows():
    print(f"  [Ref={r['sim_reference_vector']:.2f}, Cent={r['sim_score_centroid']:.2f}] {r['appln_title']}")

print(f"\n=== 4. DISAGREEMENTS (Centroid >= 0.78 but Ref < 0.74 | Total: {len(only_cent):,}) ===")
for _, r in only_cent.sample(min(5, len(only_cent)), random_state=42).iterrows():
    print(f"  [Cent={r['sim_score_centroid']:.2f}, Ref={r['sim_reference_vector']:.2f}] {r['appln_title']}")

=== 1. SCORE SEPARATION: CORE UAV CODES vs. KEYWORD-ONLY FAMILIES ===

[sim_score_centroid]
                  n    0.1   0.25    0.5   0.75    0.9
in_core_code                                          
False         55433 -0.064  0.396  0.607  0.740  0.845
True          39993  0.420  0.581  0.718  0.831  0.898

[sim_reference_vector]
                  n    0.1   0.25    0.5   0.75    0.9
in_core_code                                          
False         55433 -0.013  0.405  0.629  0.751  0.840
True          39993  0.414  0.577  0.709  0.811  0.875

Pearson correlation between Strategy A and Strategy B: 0.988


/tmp/ipykernel_358/101927007.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_aero = df['cpc_codes'].fillna('').str.contains(aero_pattern, regex=True)
/tmp/ipykernel_358/101927007.py:22: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_noise = df['cpc_codes'].fillna('').str.contains(noise_pattern, regex=True)



=== 2. THRESHOLD SWEEP COMPARISON ===
 Threshold  Cent_N  Cent_Aero%  Cent_Noise%  Ref_N  Ref_Aero%  Ref_Noise%  Mean_N  Mean_Aero%  Jaccard
      0.68   43090        71.4          6.9  44734       68.8        10.0   43849        70.3     0.85
      0.70   39362        72.9          6.1  40483       70.3         8.9   39890        71.8     0.83
      0.72   35651        74.3          5.3  36263       71.9         8.0   35904        73.3     0.82
      0.74   32008        75.6          4.5  32061       73.4         7.0   31931        74.8     0.80
      0.76   28422        77.0          3.9  27983       74.9         6.1   28065        76.2     0.79
      0.78   24846        78.2          3.3  23894       76.3         5.1   24339        77.5     0.77
      0.80   21464        79.2          2.8  20106       77.7         4.5   20622        78.7     0.75
      0.82   18252        80.0          2.2  16352       79.0         3.8   17145        79.6     0.72
      0.84   15041        81.1    

## 4. Apply Selected Filter & Run CPC Subclass Migration Audit
Set `FILTER_COL` (`"sim_score_centroid"`, `"sim_reference_vector"`, or `"sim_mean"`) and `SIM_THRESHOLD` based on the sweep above, then verify domain consolidation across CPC subclasses.

In [17]:
# Configure chosen similarity metric and cutoff threshold
FILTER_COL = "sim_mean"
SIM_THRESHOLD = 0.78  # or 0.76 if sharing via Git CLI / Google Drive

df_filtered = df[df[FILTER_COL] >= SIM_THRESHOLD].copy().reset_index(drop=True)
print(f"Retained {len(df_filtered):,} patent families out of {len(df):,} ({len(df_filtered)/len(df):.1%}) using {FILTER_COL} >= {SIM_THRESHOLD}")

# Extract 4-character CPC subclasses and compare pre- vs. post-filtering distributions
def extract_cpc_subclasses(cpc_series):
    all_codes = cpc_series.dropna().astype(str).str.findall(r'\b[A-H]\d{2}[A-Z]\b')
    flattened = [code for sublist in all_codes for code in set(sublist)]
    return pd.Series(flattened)

baseline_counts = extract_cpc_subclasses(df['cpc_codes']).value_counts(normalize=True) * 100
filtered_counts = extract_cpc_subclasses(df_filtered['cpc_codes']).value_counts(normalize=True) * 100

migration_df = pd.DataFrame({
    'Baseline Share (%)': baseline_counts,
    'Filtered Share (%)': filtered_counts
}).fillna(0)

migration_df['Share Change (pp)'] = migration_df['Filtered Share (%)'] - migration_df['Baseline Share (%)']
migration_df = migration_df.sort_values(by='Filtered Share (%)', ascending=False)

core_5 = ['B64U', 'B64C', 'B64D', 'G05D', 'B64F']
base_top5 = migration_df.loc[migration_df.index.isin(core_5), 'Baseline Share (%)'].sum()
filt_top5 = migration_df.loc[migration_df.index.isin(core_5), 'Filtered Share (%)'].sum()

print(f"\nCore 5 Aeronautical Subclasses Share: {base_top5:.1f}% (Baseline) -> {filt_top5:.1f}% (Filtered)")
print("\n=== CPC Subclass Migration Audit (Top 15 Filtered Subclasses) ===")
print(migration_df.head(15).round(3).to_string())

Retained 24,339 patent families out of 95,426 (25.5%) using sim_mean >= 0.78

Core 5 Aeronautical Subclasses Share: 43.7% (Baseline) -> 71.5% (Filtered)

=== CPC Subclass Migration Audit (Top 15 Filtered Subclasses) ===
      Baseline Share (%)  Filtered Share (%)  Share Change (pp)
B64C              11.703              20.025              8.321
B64U              12.121              19.567              7.446
B64D               9.305              13.980              4.675
G05D               8.100              12.156              4.056
B64F               2.470               5.726              3.255
G08G               2.095               2.105              0.010
B60L               1.154               1.660              0.507
G01S               2.436               1.596             -0.840
G01C               2.378               1.420             -0.958
G05B               0.783               0.911              0.128
H02J               0.916               0.807             -0.109
B60P        

## 5. Export Finalized Stage 2 Dataset
Saves the filtered patent family dataset to Parquet for team sharing, manual validation sampling (Notebook 3), and Stage 3 landscape analysis.

In [18]:
df_final = df_filtered.reset_index(drop=True)
output_parquet = "flying_drones_stage2_clean_final.parquet"
df_final.to_parquet(output_parquet, index=False)

size_mb = os.path.getsize(output_parquet) / (1024 * 1024)
print(f"Successfully saved '{output_parquet}': {len(df_final):,} patent families ({size_mb:.2f} MB)")

Successfully saved 'flying_drones_stage2_clean_final.parquet': 24,339 patent families (24.26 MB)
